# Monte Carlo Gacha + XP Simulator v4 — RAM-efficient
Batched seeds, streamed CSV output, chunked merging, minimal in-RAM footprint.


In [1]:
# =============================================================
# 1) Imports, display opts, and quick instructions
# =============================================================
# Run cells top to bottom. Edit CONFIG first.
# This version is RAM-efficient:
# - Runs in batches (max 400 players per batch) across multiple RNG seeds
# - Streams rows directly to CSV to avoid holding big DataFrames in memory
# - Merges CSVs and computes representatives via chunked reads
# - Loads only the representative players' rows to render charts
#
# Tip: If you still hit RAM, set CONFIG["dump_xp"]=False to drop the 'xp' column,
# and the XP>0 rarity charts will be skipped automatically.

import numpy as np
import pandas as pd
import altair as alt
from pathlib import Path
import json
import dataclasses
import collections
from typing import Dict, List, Tuple, Iterable
from copy import deepcopy
import yaml
import zipfile

alt.data_transformers.disable_max_rows()


# =============================================================
# 2) CONFIG: single source of truth
# =============================================================
CONFIG = {
    # Randomness + batching
    "rng_seed": 42,
    "total_players": 1200,       # total Monte Carlo players to simulate
    "batch_max_players": 300,    # runs in batches ≤ this size for RAM safety

    "num_weeks": 104,

    # Gacha
    "drop_probs": {
        "Common": 0.00,
        "Uncommon": 0.49,
        "Rare": 0.45,
        "Epic": 0.055,
        "Legendary": 0.005,
    },
    # Starting pool per season before additions
    "pool_sizes_initial": {
        "Common": 20,
        "Uncommon": 40,
        "Rare": 30,
        "Epic": 25,
        "Legendary": 20,
    },
    # Per-season additions relative to season 1
    # Season s has: base + additions * (s-1). Example: Epic 25 + 5*(s-1)
    "pool_additions_every_4_weeks": {
        "Common": 0,
        "Uncommon": 0,
        "Rare": 0,
        "Epic": 5,
        "Legendary": 7,
    },
    "pulls_per_week": 7,

    # XP
    "xp_per_week": 1_500_000,
    "xp_pct_seasonal": 0.80,      # remainder to non-seasonal
    "seasonal_targets_per_week": 4,
    "nonseasonal_targets_per_week": 8,
    "rarity_weights": {
        "Common": 1,
        "Uncommon": 2,
        "Rare": 3,
        "Epic": 4,
        "Legendary": 5,
    },

    # Level cap and XP table (cumulative_xp is XP before reaching that level)
    "level_cap": 49,
    "xp_table": [
        {"level": 1,  "xp_cost": 1000,   "cumulative_xp": 0},
        {"level": 2,  "xp_cost": 1700,   "cumulative_xp": 1000},
        {"level": 3,  "xp_cost": 2625,   "cumulative_xp": 2700},
        {"level": 4,  "xp_cost": 3725,   "cumulative_xp": 5325},
        {"level": 5,  "xp_cost": 5000,   "cumulative_xp": 9050},
        {"level": 6,  "xp_cost": 6450,   "cumulative_xp": 14050},
        {"level": 7,  "xp_cost": 8050,   "cumulative_xp": 20500},
        {"level": 8,  "xp_cost": 8925,   "cumulative_xp": 28550},
        {"level": 9,  "xp_cost": 10750,  "cumulative_xp": 37475},
        {"level": 10, "xp_cost": 12725,  "cumulative_xp": 48225},
        {"level": 11, "xp_cost": 14875,  "cumulative_xp": 60950},
        {"level": 12, "xp_cost": 18000,  "cumulative_xp": 75825},
        {"level": 13, "xp_cost": 20550,  "cumulative_xp": 93825},
        {"level": 14, "xp_cost": 23250,  "cumulative_xp": 114375},
        {"level": 15, "xp_cost": 24650,  "cumulative_xp": 137625},
        {"level": 16, "xp_cost": 27575,  "cumulative_xp": 162275},
        {"level": 17, "xp_cost": 30650,  "cumulative_xp": 189850},
        {"level": 18, "xp_cost": 33875,  "cumulative_xp": 220500},
        {"level": 19, "xp_cost": 37250,  "cumulative_xp": 254375},
        {"level": 20, "xp_cost": 40750,  "cumulative_xp": 291625},
        {"level": 21, "xp_cost": 42575,  "cumulative_xp": 332375},
        {"level": 22, "xp_cost": 46300,  "cumulative_xp": 374950},
        {"level": 23, "xp_cost": 52700,  "cumulative_xp": 421250},
        {"level": 24, "xp_cost": 56900,  "cumulative_xp": 473950},
        {"level": 25, "xp_cost": 61275,  "cumulative_xp": 530850},
        {"level": 26, "xp_cost": 65800,  "cumulative_xp": 592125},
        {"level": 27, "xp_cost": 70475,  "cumulative_xp": 657925},
        {"level": 28, "xp_cost": 76500,  "cumulative_xp": 728400},
        {"level": 29, "xp_cost": 81650,  "cumulative_xp": 804900},
        {"level": 30, "xp_cost": 86950,  "cumulative_xp": 886550},
        {"level": 31, "xp_cost": 92400,  "cumulative_xp": 973500},
        {"level": 32, "xp_cost": 98000,  "cumulative_xp": 1065900},
        {"level": 33, "xp_cost": 108950, "cumulative_xp": 1163900},
        {"level": 34, "xp_cost": 115175, "cumulative_xp": 1272850},
        {"level": 35, "xp_cost": 118325, "cumulative_xp": 1388025},
        {"level": 36, "xp_cost": 124775, "cumulative_xp": 1506350},
        {"level": 37, "xp_cost": 131400, "cumulative_xp": 1631125},
        {"level": 38, "xp_cost": 138175, "cumulative_xp": 1762525},
        {"level": 39, "xp_cost": 152375, "cumulative_xp": 1900700},
        {"level": 40, "xp_cost": 159825, "cumulative_xp": 2053075},
        {"level": 41, "xp_cost": 163600, "cumulative_xp": 2212900},
        {"level": 42, "xp_cost": 171300, "cumulative_xp": 2376500},
        {"level": 43, "xp_cost": 179175, "cumulative_xp": 2547800},
        {"level": 44, "xp_cost": 216225, "cumulative_xp": 2726975},
        {"level": 45, "xp_cost": 273100, "cumulative_xp": 2943200},
        {"level": 46, "xp_cost": 344600, "cumulative_xp": 3216300},
        {"level": 47, "xp_cost": 434425, "cumulative_xp": 3560900},
        {"level": 48, "xp_cost": 487625, "cumulative_xp": 3995325},
        {"level": 49, "xp_cost": 547200, "cumulative_xp": 4482950},
    ],

    # Chart/outputs
    "rpg_rarity_colors": {
        "Common": "#BFBFBF",
        "Uncommon": "#1EFF00",
        "Rare": "#0070FF",
        "Epic": "#A335EE",
        "Legendary": "#FF8000",
    },
    "save_dir": "outputs_v4",
    "dump_xp": True,   # set False to drop 'xp' and skip XP>0 charts
}
BASE_CONFIG = deepcopy(CONFIG)


# =============================================================
# 3) Validation and derived values
# =============================================================
RARITY_ORDER = {"Legendary": 5, "Epic": 4, "Rare": 3, "Uncommon": 2, "Common": 1}

def validate_config(cfg: dict) -> None:
    p_sum = round(sum(cfg["drop_probs"].values()), 12)
    if abs(p_sum - 1.0) > 1e-9:
        raise ValueError(f"Drop probabilities must sum to 1. Got {p_sum}")
    ks = set(cfg["drop_probs"].keys())
    for k in ["pool_sizes_initial", "pool_additions_every_4_weeks", "rarity_weights"]:
        if set(cfg[k].keys()) != ks:
            raise ValueError(f"Keys mismatch for {k}. Expected {ks}.")
    if cfg["level_cap"] != max(row["level"] for row in cfg["xp_table"]):
        raise ValueError("level_cap must match highest level in xp_table")
    if cfg["pulls_per_week"] < 0 or cfg["num_weeks"] <= 0 or cfg["total_players"] <= 0:
        raise ValueError("Counts must be positive")
    if not (0 <= cfg["xp_pct_seasonal"] <= 1):
        raise ValueError("xp_pct_seasonal must be in [0,1]")

def season_index(week: int) -> int:
    return (week - 1) // 4

validate_config(CONFIG)
RNG = np.random.default_rng(CONFIG["rng_seed"])


# =============================================================
# 4) XP table + level lookup
# =============================================================
def build_xp_table(cfg: dict) -> pd.DataFrame:
    df = pd.DataFrame(cfg["xp_table"]).copy().sort_values("level").reset_index(drop=True)
    df.rename(columns={"cumulative_xp": "cum_xp_start"}, inplace=True)
    df["cum_xp_end"] = df["cum_xp_start"].shift(-1, fill_value=np.inf).astype(float)
    return df

def level_from_xp(total_xp: int, cum_starts: np.ndarray, level_cap: int) -> int:
    idx = int(np.searchsorted(cum_starts, total_xp, side="right"))
    return min(idx, level_cap)

xp_df = build_xp_table(CONFIG)
CUM_STARTS = xp_df["cum_xp_start"].to_numpy()


# =============================================================
# 5) Season-unique hero registry
# =============================================================
def _id_prefix(rarity: str) -> str:
    return {"Legendary": "L", "Epic": "E", "Rare": "R", "Uncommon": "U", "Common": "C"}[rarity]

def _season_count(rarity: str, s_idx: int, cfg: dict) -> int:
    base = cfg["pool_sizes_initial"][rarity]
    add = cfg["pool_additions_every_4_weeks"][rarity]
    return base + add * s_idx

def init_hero_registry(cfg: dict) -> pd.DataFrame:
    rows = []
    last_season_idx = season_index(cfg["num_weeks"])
    for s_idx in range(0, last_season_idx + 1):
        season_start_week = s_idx * 4 + 1
        for rarity in cfg["pool_sizes_initial"].keys():
            n_this = _season_count(rarity, s_idx, cfg)
            pref = _id_prefix(rarity)
            for i in range(1, n_this + 1):
                rows.append({
                    "hero_id": f"{pref}-{i:03d}_S{s_idx+1:02d}",
                    "rarity": rarity,
                    "season_idx": s_idx,
                    "introduced_week": season_start_week,
                })
    return pd.DataFrame(rows)

def pool_for_week(week: int, registry: pd.DataFrame) -> Dict[str, np.ndarray]:
    s_idx = season_index(week)
    avail = registry[registry["season_idx"] == s_idx]
    out = {r: avail[avail["rarity"] == r]["hero_id"].to_numpy() for r in CONFIG["drop_probs"].keys()}
    for r in CONFIG["drop_probs"].keys():
        out.setdefault(r, np.array([], dtype=str))
    return out

REGISTRY = init_hero_registry(CONFIG)


# =============================================================
# 6) RNG & pulls
# =============================================================
def sample_rarities(n_pulls: int, drop_probs: Dict[str, float]) -> np.ndarray:
    rarities = np.array(list(drop_probs.keys()))
    probs = np.array(list(drop_probs.values()), dtype=float)
    return RNG.choice(rarities, size=n_pulls, p=probs)

def sample_hero_id(rarity: str, pool: Dict[str, np.ndarray]) -> str:
    return str(RNG.choice(pool[rarity]))


# =============================================================
# 7) Player / state
# =============================================================
@dataclasses.dataclass
class HeroState:
    rarity: str
    xp: int
    level: int
    acquired_week: int

class PlayerState:
    def __init__(self, player_id: int):
        self.player_id = player_id
        self.owned: Dict[str, HeroState] = {}

# Global target log buffer, flushed per player
TARGET_LOG = []


# =============================================================
# 8) Weekly pulls
# =============================================================
def do_weekly_pulls(state: PlayerState, week: int, pool: Dict[str, np.ndarray], pulls_per_week: int, drop_probs: Dict[str, float]):
    rarities = sample_rarities(pulls_per_week, drop_probs)
    for r in rarities:
        if pool[r].size == 0:
            continue
        hid = sample_hero_id(r, pool)
        if hid not in state.owned:
            state.owned[hid] = HeroState(rarity=r, xp=0, level=1, acquired_week=week)


# =============================================================
# 9) Target selection
# =============================================================
def select_targets(state: PlayerState, week: int, seasonal_k: int, nonseasonal_k: int) -> Tuple[List[str], List[str]]:
    s_idx = season_index(week)
    seasonal, nonseasonal = [], []
    for hid, h in state.owned.items():
        is_seasonal = season_index(h.acquired_week) == s_idx
        (seasonal if is_seasonal else nonseasonal).append((hid, h))
    key = lambda t: (-RARITY_ORDER[t[1].rarity], -t[1].level, t[1].acquired_week, t[0])
    seasonal_ids = [hid for hid, _ in sorted(seasonal, key=key)[:seasonal_k]]
    nonseasonal_ids = [hid for hid, _ in sorted(nonseasonal, key=key)[:nonseasonal_k]]
    return seasonal_ids, nonseasonal_ids

def select_targets_logged(state: PlayerState, week: int, seasonal_k: int, nonseasonal_k: int):
    seasonal_ids, nonseasonal_ids = select_targets(state, week, seasonal_k, nonseasonal_k)
    for hid in seasonal_ids:
        h = state.owned[hid]
        TARGET_LOG.append({"player_id": state.player_id, "week": week, "hero_id": hid, "rarity": h.rarity, "target_type": "seasonal"})
    for hid in nonseasonal_ids:
        h = state.owned[hid]
        TARGET_LOG.append({"player_id": state.player_id, "week": week, "hero_id": hid, "rarity": h.rarity, "target_type": "nonseasonal"})
    return seasonal_ids, nonseasonal_ids


# =============================================================
# 10) XP allocation
# =============================================================
def allocate_xp_category(state: PlayerState, hero_ids: List[str], xp_budget: int,
                          rarity_weights: Dict[str, int], cum_starts: np.ndarray, level_cap: int) -> int:
    if not hero_ids or xp_budget <= 0:
        return xp_budget
    counts = collections.Counter(state.owned[h].rarity for h in hero_ids)
    weights = {r: counts[r] * rarity_weights[r] for r in counts}
    total_w = sum(weights.values())
    if total_w <= 0:
        return xp_budget

    remainder = int(xp_budget)
    active = set(hero_ids)

    while remainder > 0 and active:
        per_rarity = {r: int(remainder * weights[r] / total_w) for r in weights}
        if sum(per_rarity.values()) == 0:
            top_r = max(per_rarity.keys(), key=lambda rr: RARITY_ORDER[rr])
            per_rarity[top_r] = remainder
        spent = 0
        for r, share in per_rarity.items():
            ids_r = [h for h in active if state.owned[h].rarity == r]
            if not ids_r or share <= 0:
                continue
            per_hero = share // len(ids_r)
            extra = share - per_hero * len(ids_r)
            for i, h in enumerate(ids_r):
                give = per_hero + (1 if i < extra else 0)
                if give <= 0:
                    continue
                hero = state.owned[h]
                if hero.level >= level_cap:
                    continue
                new_xp = hero.xp + give
                new_lvl = level_from_xp(new_xp, cum_starts, level_cap)
                hero.xp = int(new_xp)
                hero.level = int(new_lvl)
                spent += give
            active = {h for h in active if state.owned[h].level < level_cap}
        if spent == 0:
            break
        remainder -= spent
    return remainder

def apply_weekly_xp(state: PlayerState, week: int, cfg: dict, cum_starts: np.ndarray) -> int:
    xp_total = int(cfg["xp_per_week"])
    xp_seasonal = int(xp_total * cfg["xp_pct_seasonal"])
    xp_non = xp_total - xp_seasonal
    seasonal_ids, nonseasonal_ids = select_targets_logged(
        state, week, cfg["seasonal_targets_per_week"], cfg["nonseasonal_targets_per_week"]
    )
    wasted_s = allocate_xp_category(state, seasonal_ids, xp_seasonal, cfg["rarity_weights"], cum_starts, cfg["level_cap"])
    wasted_n = allocate_xp_category(state, nonseasonal_ids, xp_non, cfg["rarity_weights"], cum_starts, cfg["level_cap"])
    return int(wasted_s + wasted_n)


# =============================================================
# 11) Snapshot + stream-to-CSV writers
# =============================================================
SNAP_COLS_WITH_XP = ["player_id","week","hero_id","rarity","level","xp","acquired_week","season_idx","is_seasonal_week"]
SNAP_COLS_NO_XP   = ["player_id","week","hero_id","rarity","level","acquired_week","season_idx","is_seasonal_week"]

def snapshot_week(state: PlayerState, week: int, dump_xp: bool) -> List[dict]:
    out = []
    s_idx = season_index(week)
    for hid, h in state.owned.items():
        row = {
            "player_id": state.player_id,
            "week": week,
            "hero_id": hid,
            "rarity": h.rarity,
            "level": int(h.level),
            "acquired_week": int(h.acquired_week),
            "season_idx": int(season_index(h.acquired_week)),
            "is_seasonal_week": int(season_index(h.acquired_week) == s_idx),
        }
        if dump_xp:
            row["xp"] = int(h.xp)
        out.append(row)
    return out

def write_rows_csv(path: Path, rows: List[dict], header_written: bool):
    if not rows:
        return header_written
    df = pd.DataFrame.from_records(rows)
    df.to_csv(path, mode="a", index=False, header=not header_written)
    return True

def flush_target_log_csv(path: Path, header_written: bool):
    global TARGET_LOG
    if not TARGET_LOG:
        return header_written
    df = pd.DataFrame(TARGET_LOG)
    df.to_csv(path, mode="a", index=False, header=not header_written)
    TARGET_LOG = []
    return True


# =============================================================
# 12) Simulate a batch directly to CSV
# =============================================================
def simulate_batch_to_csv(batch_players: int, cfg: dict, registry: pd.DataFrame, cum_starts: np.ndarray,
                          out_csv: Path, targets_csv: Path, player_id_offset: int = 0):
    header_written = out_csv.exists()
    t_header_written = targets_csv.exists()

    for local_pid in range(batch_players):
        pid = player_id_offset + local_pid
        state = PlayerState(pid)
        for week in range(1, cfg["num_weeks"] + 1):
            pool = pool_for_week(week, registry)
            do_weekly_pulls(state, week, pool, cfg["pulls_per_week"], cfg["drop_probs"])
            _ = apply_weekly_xp(state, week, cfg, cum_starts)
            rows = snapshot_week(state, week, dump_xp=cfg["dump_xp"])
            header_written = write_rows_csv(out_csv, rows, header_written)
        # flush per-player target log to disk to keep RAM low
        t_header_written = flush_target_log_csv(targets_csv, t_header_written)


# =============================================================
# 13) Batch driver
# =============================================================

def run_scenario():
    run_all_batches()

    # Representatives from merged CSV, then load only their rows and chart
    final_week = CONFIG["num_weeks"]
    reps = choose_representatives_from_merged(MERGED_PLAYERS_CSV, final_week)
    print("Representatives:", reps)
    with open(SAVE_DIR / "representatives.json","w") as f:
        json.dump(reps, f, indent=2)

    # Collect rows for representatives via chunked read
    rep_ids = set(reps.values())
    cols_needed = ["player_id","week","hero_id","rarity","level","acquired_week","season_idx","is_seasonal_week"]
    if CONFIG["dump_xp"]:
        cols_needed.append("xp")
    rep_rows = []
    for chunk in pd.read_csv(MERGED_PLAYERS_CSV, usecols=cols_needed, chunksize=1_000_000):
        sub = chunk[chunk["player_id"].isin(rep_ids)]
        if not sub.empty:
            rep_rows.append(sub)
    rep_df = pd.concat(rep_rows, ignore_index=True) if rep_rows else pd.DataFrame(columns=cols_needed)

    # Chart order: mode → median → p75
    ordered = [("mode", reps["mode"]), ("median", reps["median"]), ("p75", reps["p75"])]
    sc_name = CONFIG.get("scenario_name", "")
    title_prefix = f"{sc_name} - " if sc_name else ""
    for name, pid in ordered:
        dfp = rep_df[rep_df["player_id"] == pid].copy()

        # A) Rarity progression (XP>0 if available, else owned)
        counts_xp = build_rarity_counts_from_df(dfp, CONFIG["num_weeks"])
        chart_A = make_collection_by_rarity_chart(
            counts_xp, CONFIG["rpg_rarity_colors"], f"{title_prefix}[{name}] Rarity progression ({'XP > 0' if CONFIG['dump_xp'] else 'all owned'})"
        )
        display(chart_A); save_chart(chart_A, SAVE_DIR / f"{name}_A_rarity_progression.html")

        # B1) Seasonal target composition
        comp_seasonal = build_target_comp_from_log_chunked(MERGED_TARGETS_CSV, pid, CONFIG["num_weeks"], kind="seasonal")
        chart_B1 = make_stacked_bar_with_labels(comp_seasonal, CONFIG["rpg_rarity_colors"], f"{title_prefix}[{name}] Seasonal target composition")
        display(chart_B1); save_chart(chart_B1, SAVE_DIR / f"{name}_B1_seasonal_targets.html")

        # B2) Non-seasonal target composition
        comp_nonseasonal = build_target_comp_from_log_chunked(MERGED_TARGETS_CSV, pid, CONFIG["num_weeks"], kind="nonseasonal")
        chart_B2 = make_stacked_bar_with_labels(comp_nonseasonal, CONFIG["rpg_rarity_colors"], f"{title_prefix}[{name}] Non-seasonal target composition")
        display(chart_B2); save_chart(chart_B2, SAVE_DIR / f"{name}_B2_nonseasonal_targets.html")

        # C) Lines
        chart_C = make_line_chart(dfp, CONFIG["rpg_rarity_colors"], f"{title_prefix}[{name}] Weekly Collection Status")
        display(chart_C); save_chart(chart_C, SAVE_DIR / f"{name}_C_lines.html")

        # D) Faceted
        chart_D = make_facet_chart_by_rarity(dfp, CONFIG["rpg_rarity_colors"], f"{title_prefix}[{name}] Weekly Collection Status (Faceted)")
        display(chart_D); save_chart(chart_D, SAVE_DIR / f"{name}_D_facet.html")

    # QA spot checks
    assert abs(sum(CONFIG["drop_probs"].values()) - 1.0) < 1e-9
    w1 = pool_for_week(1, REGISTRY); w5 = pool_for_week(5, REGISTRY)
    for r in CONFIG["drop_probs"].keys():
        base_n = CONFIG["pool_sizes_initial"][r]; add_n = CONFIG["pool_additions_every_4_weeks"][r]
        assert len(w1[r]) == base_n + add_n * 0
        assert len(w5[r]) == base_n + add_n * 1
    print("QA checks passed.")
    # Remove raw CSVs and helper files to keep only charts
    for _p in [MERGED_PLAYERS_CSV, MERGED_TARGETS_CSV, SAVE_DIR / "representatives.json"]:
        if _p.exists():
            _p.unlink()

# =============================================================
# 16) Run scenarios from YAML
# =============================================================

def deep_update(base, updates):
    for k, v in updates.items():
        if isinstance(v, dict) and isinstance(base.get(k), dict):
            deep_update(base[k], v)
        else:
            base[k] = v
    return base

scenarios = yaml.safe_load(open("scenarios.yaml"))
results_root = Path("runs")
results_root.mkdir(exist_ok=True)

for sc in scenarios:
    CONFIG = deepcopy(BASE_CONFIG)
    deep_update(CONFIG, sc.get("CONFIG", {}))
    CONFIG["scenario_name"] = sc["name"]
    SAVE_DIR = results_root / sc["name"]
    CONFIG["save_dir"] = str(SAVE_DIR)
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    MERGED_PLAYERS_CSV = SAVE_DIR / "player_weekly_merged.csv"
    MERGED_TARGETS_CSV = SAVE_DIR / "targets_log_merged.csv"
    run_scenario()

with zipfile.ZipFile("all_results.zip", "w") as z:
    for p in results_root.rglob("*"):
        if p.is_file():
            z.write(p, arcname=p.relative_to(results_root))


Output hidden; open in https://colab.research.google.com to view.